After going through all the other notebooks, we can derive the final production data and use it to correct import, i.e., remove re-exports from them.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('C://Users/11max/PycharmProjects/Regioinvent/src/')
import regioinvent
import sqlite3

Just initialize the Regioinvent class to get all the matching dicts and stuff

In [2]:
self = regioinvent.Regioinvent(bw_project_name='ecoinvent3.12', ecoinvent_database_name='ecoinvent-3.12-cutoff', ecoinvent_version='3.12')

Connecting to the SQL database with trade information

In [3]:
trade_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

Load non-corrected imports

In [4]:
import_data = pd.read_sql('SELECT * FROM [Import data]', trade_conn)

Load net exports

In [5]:
net_exports_data = pd.read_sql('SELECT * FROM [Export data]', trade_conn)

Get all the production data

In [7]:
# load real production volumes data
real_domestic_prod = pd.read_sql('SELECT * FROM "Real production data"', con=trade_conn)
# change type to float and not string
real_domestic_prod.loc[:,'quantity (t)'] = real_domestic_prod.loc[:,'quantity (t)'].astype(float)
real_domestic_prod = real_domestic_prod.groupby(
    ['cmdCode', 'producer', 'source', 'refYear']).sum().reset_index()
# convert to ISO 2 letter codes
real_domestic_prod.producer = [self.convert_ecoinvent_geos[i] for i in real_domestic_prod.producer]
# identify the commodity codes for which we have actual production data and full country coverage
products_with_real_prod_data_and_full_coverage = set(real_domestic_prod.loc[real_domestic_prod.country_coverage == 'complete'].cmdCode)
# identify the commodity codes for which we have actual production data but a partial country coverage (like with PRODCOM data)
products_with_real_prod_data_and_partial_coverage = list(zip(real_domestic_prod.loc[real_domestic_prod.country_coverage == 'partial',
                                                             ['cmdCode','producer']].drop_duplicates().cmdCode,
                                                             real_domestic_prod.loc[real_domestic_prod.country_coverage == 'partial', 
                                                             ['cmdCode', 'producer']].drop_duplicates().producer))
# rename column producer to exporter
real_domestic_prod = real_domestic_prod.rename(columns={'producer': 'exporter'})

In [8]:
# load estimates from EXIOBASE
domestic_prod_ratios = pd.read_sql('SELECT * FROM "Domestic use ratios exiobase"', con=trade_conn)
# domestic production is estimated from net exports so copy them
estimated_domestic_prod = net_exports_data.copy('deep')
estimated_domestic_prod.loc[:, 'COMTRADE_reporter'] = estimated_domestic_prod.exporter.copy()
# go from ISO3 codes to country codes for respective databases
net_exports_data.exporter = [self.convert_ecoinvent_geos[i] for i in net_exports_data.exporter]
estimated_domestic_prod.exporter = [self.convert_ecoinvent_geos[i] for i in estimated_domestic_prod.exporter]
estimated_domestic_prod.COMTRADE_reporter = [self.convert_exiobase_geos[i] for i in
                                             estimated_domestic_prod.COMTRADE_reporter]
# add HS codes
estimated_domestic_prod = estimated_domestic_prod.merge(
    pd.DataFrame.from_dict(self.hs_class_to_exio, orient='index', columns=['cmdExio']).reset_index().rename(
        columns={'index': 'cmdCode'}))
# merge with domestic production ratios
estimated_domestic_prod = estimated_domestic_prod.merge(domestic_prod_ratios,
                                                        left_on=['COMTRADE_reporter', 'cmdExio'],
                                                        right_on=['country', 'commodity'], how='left')
# apply the ratios to determine domestic production
estimated_domestic_prod.loc[:, 'quantity (t)'] = (
        estimated_domestic_prod.loc[:, 'quantity (t)'] /
        (1 - estimated_domestic_prod.loc[:, 'domestic use (%)'] / 100) *
        estimated_domestic_prod.loc[:, 'domestic use (%)'] / 100)
# add importer column equal to export column (i.e., domestic production)
estimated_domestic_prod.loc[:, 'importer'] = estimated_domestic_prod.exporter
# some infinites came up, replace them by zero
estimated_domestic_prod.loc[:, 'quantity (t)'] = estimated_domestic_prod.loc[:, 'quantity (t)'].replace(np.inf, 0.0)
# only keep non-null values
estimated_domestic_prod = estimated_domestic_prod.loc[estimated_domestic_prod.loc[:, 'quantity (t)'] != 0]

# only keep values for products with no real production data (partial coverage)
estimated_domestic_prod = estimated_domestic_prod.loc[[i for i in estimated_domestic_prod.index if 
                                                       (estimated_domestic_prod.loc[i, 'cmdCode'], estimated_domestic_prod.loc[i, 'exporter']) not in
                                                       products_with_real_prod_data_and_partial_coverage]]
# only keep values for products with no real production data (complete coverage)
estimated_domestic_prod = estimated_domestic_prod.loc[[i for i in estimated_domestic_prod.index if
                                                       estimated_domestic_prod.loc[
                                                           i, 'cmdCode'] not in products_with_real_prod_data_and_full_coverage],
                                                      import_data.columns]
# add source information
estimated_domestic_prod.loc[:, 'source'] = 'EXIOBASE - rough estimate from EXIOBASE 3.9.5'

In [10]:
# concatenate domestic all production data (real and estimated through exiobase)
domestic_production = pd.concat([real_domestic_prod, estimated_domestic_prod])

In [11]:
# concatenate net exports and domestic data into production data
production_data = pd.concat([net_exports_data, domestic_production.drop(['importer'], axis=1)]).drop(['source', 'country_coverage'], axis=1)
production_data = production_data.groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()

Now we need to correct import trade data, because it is still relying on re-exports. For each import, we assert if the originating country is a producer or not (through comparison with production_data). If it is, then nothing is changed. If it is not identified as a producing country, then we redistribute this import to the 5 biggest producing countries of that commodity.

In [14]:
# Get a list of actual producers per commodity
actual_producers = production_data[production_data['quantity (t)'] != 0].groupby('cmdCode')['exporter'].unique().to_dict()

# Get top 5 exporters and their shares via vectorization
exporter_totals = net_exports_data.groupby(['cmdCode', 'exporter'])['quantity (t)'].sum().reset_index()

# Sort and get top 5 per commodity
top_5 = exporter_totals.sort_values(['cmdCode', 'quantity (t)'], ascending=[True, False])
top_5 = top_5.groupby('cmdCode').head(5).copy()

# Calculate market share within those top 5
top_5['share'] = top_5.groupby('cmdCode')['quantity (t)'].transform(lambda x: x / x.sum())

# Create the lookup dictionary for the second step
biggest_exporters = top_5.groupby('cmdCode').apply(
    lambda x: list(zip(x['exporter'], x['share'])), include_groups=False
).to_dict()

In [15]:
# Separate valid imports from re-exports
is_producer = import_data.apply(lambda x: x['exporter'] in actual_producers.get(x['cmdCode'], []), axis=1)

valid_imports = import_data[is_producer].copy()
re_exports = import_data[~is_producer].copy()

# Map the top 5 producers to the re_exports
# This creates 5 rows for every 1 re-export row
re_exports_expanded = re_exports.merge(top_5[['cmdCode', 'exporter', 'share']], on='cmdCode', suffixes=('_old', ''))

# Adjust quantities and cleanup
re_exports_expanded['quantity (t)'] = re_exports_expanded['quantity (t)'] * re_exports_expanded['share']
re_exports_expanded = re_exports_expanded.drop(columns=['exporter_old', 'share'])

# Final Combine (Only one concat operation!)
corrected_import_data = pd.concat([valid_imports, re_exports_expanded], ignore_index=True)

In [16]:
corrected_import_data = corrected_import_data.groupby(['cmdCode', 'refYear', 'importer', 'exporter', 'source']).sum().reset_index()

Replace the import data in the SQL database by the corrected imports

In [19]:
cursor = trade_conn.cursor()
cursor.execute('DROP TABLE [Import data]')

corrected_import_data.loc[corrected_import_data.loc[:,'quantity (t)'] != 0].set_index('cmdCode').to_sql('Import data', trade_conn)

8266353

Also write the concatenated domestic production data in the SQL database

In [26]:
cursor = trade_conn.cursor()
cursor.execute('DROP TABLE [Domestic production data]')

domestic_production.loc[domestic_production.loc[:,'quantity (t)'] != 0].set_index('cmdCode').drop(['importer', 'country_coverage'], axis=1).to_sql('Domestic production data', trade_conn)

150411

In [11]:
# also change export data table to use 2-letter ISO country codes instead of 3-letter codes
cursor = trade_conn.cursor()
cursor.execute('DROP TABLE [Export data]')

net_exports_data.loc[net_exports_data.loc[:,'quantity (t)'] != 0].set_index('cmdCode').to_sql('Export data', trade_conn)

145208